# 모델 학습 코드

- 대상 모델:
    - xLAM-2-8b-fc-r
- 학습 데이터:
    - 파일명:
    - 데이터 수: 26213개
    - 형식: min
    - 액션: 5,243개 (20%)

## 학습 설정

- 학습 방법: DoRA
- num_train_epochs: 200. (총 epoch 개수)
- batch_size: 8  (batch당 샘플 수)
- gradient_accumulation_steps: 8  (gradient 누적 스텝 수 , 따라서 1 batch당 effective batch size는 8 * 8 = 64)
- eval_strategy: "steps"  (평가 전략: steps 또는 epoch)
- eval_steps: 150  (평가 스텝 수)
- early_stopping_patience: 10  (10번 연속으로 validation loss가 개선되지 않으면 학습 중단)
- logging_steps: 150  (logging 스텝 수)

## 학습 환경

- GPU: A100 80GB



In [1]:
!pip install transformers==5.0.0 torch==2.10.0 accelerate==1.13.0 trl==0.29.1 peft==0.18.1 bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 49.6 MB/s eta 0:00:00


In [2]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, EarlyStoppingCallback
import peft
from peft import prepare_model_for_kbit_training
import datasets
import trl
import accelerate
import os
from google.colab import drive
from huggingface_hub import login
import time


print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"PEFT version: {peft.__version__}")
print(f"Datasets version: {datasets.__version__}")
print(f"TRL version: {trl.__version__}")
print(f"Accelerate version: {accelerate.__version__}")
print("Installation successful and modules imported.")

PyTorch version: 2.10.0+cu128
Transformers version: 5.0.0
PEFT version: 0.18.1
Datasets version: 4.0.0
TRL version: 0.29.1
Accelerate version: 1.13.0
Installation successful and modules imported.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# 경로 설정
model_path = "Salesforce/Llama-xLAM-2-8b-fc-r"
data_base_path = "/content/drive/MyDrive/TANGO2 - SDF/데이터/"
data_category_name = "최기형_양배추_20260211_20260402_v3_augmentation"
formatted_train_path = data_base_path + data_category_name + "/학습데이터/formatted_balanced_min_train_by_20_train_4motors.csv"
formatted_valid_path = data_base_path + data_category_name + "/학습데이터/formatted_balanced_min_train_by_20_valid_4motors.csv"
checkpoint_path = "/content/drive/MyDrive/TANGO2 - SDF/모델/20260415_xLAM-2-8b-fc-r_10486-row_20percent-action_min_4motors/checkpoints/"
model_save_path = "/content/drive/MyDrive/TANGO2 - SDF/모델/20260415_xLAM-2-8b-fc-r_10486-row_20percent-action_min_4motors/best_model/"

In [6]:

# 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained(model_path)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:85: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/512 [00:00<?, ?B/s]

In [9]:
from tqdm import tqdm
tqdm.pandas()
import pandas as pd
from datasets import Dataset
def batched_tokenize(batch):
    formatted_messages = []
    for i in range(len(batch['text'])):
      formatted_messages.append(batch['text'][i])
    return tokenizer(formatted_messages)

train_df = pd.read_csv(formatted_train_path)
raw_train_dataset = Dataset.from_pandas(train_df)

train_tokenized_datasets = raw_train_dataset.map(
    batched_tokenize,
    batched=True,
    remove_columns=raw_train_dataset.column_names,
    desc="Running tokenizer for min data"
    )

valid_df = pd.read_csv(formatted_valid_path)
raw_valid_dataset = Dataset.from_pandas(valid_df)

valid_tokenized_datasets = raw_valid_dataset.map(
    batched_tokenize,
    batched=True,
    remove_columns=raw_valid_dataset.column_names,
    desc="Running tokenizer for min data"
    )
# split_dataset = tokenized_datasets.train_test_split(test_size=0.1, seed=42)
# train_dataset = split_dataset['train']
# eval_dataset = split_dataset['test']


Running tokenizer for min data:   0%|          | 0/26213 [00:00<?, ? examples/s]

Running tokenizer for min data:   0%|          | 0/3276 [00:00<?, ? examples/s]

In [10]:
from peft import get_peft_model, LoraConfig
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling
# PEFT 설정
peft_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules="all-linear",
    use_dora=True,
)
model_peft = get_peft_model(model, peft_config)


save_base_path = checkpoint_path
training_args = TrainingArguments(
    output_dir=save_base_path,
    per_device_train_batch_size=64,
    gradient_accumulation_steps=1,
    num_train_epochs=200,
    save_strategy="steps",
    save_steps=150,
    save_total_limit=10,
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
)
#DataCollator 설정
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

#trainer 설정
trainer = Trainer(
    model=model_peft,
    args=training_args,
    train_dataset=train_tokenized_datasets,
    eval_dataset=valid_tokenized_datasets,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=10)],
    data_collator=data_collator,
)

In [11]:
import os
import gc
import torch
import shutil
import time
from google.colab import runtime

def clean_memory_and_disk():
    """학습 사이사이에 로컬 디스크와 메모리를 정리하는 함수"""
    gc.collect()
    torch.cuda.empty_cache()

traing_log = []
try:
    # ==========================================
    # XLAM 모델 학습
    # ==========================================
    overall_xlam_start = time.time()

    # [학습 단계]
    train_xlam_start = time.time()
    print("Starting fine-tuning for XLAM...")
    trainer.train()
    train_xlam_duration = time.time() - train_xlam_start
    print(f">>> XLAM Training Time: {train_xlam_duration:.2f} seconds")
    traing_log.append(f">>> XLAM Training Time: {train_xlam_duration:.2f} seconds")

    # [저장 단계]
    save_xlam_start = time.time()
    print("Saving XLAM model to Drive...")
    xlam_save_path = model_save_path
    model_peft.save_pretrained(xlam_save_path)
    tokenizer.save_pretrained(xlam_save_path)
    save_xlam_duration = time.time() - save_xlam_start
    print(f">>> XLAM Saving Time: {save_xlam_duration:.2f} seconds")
    traing_log.append(f">>> XLAM Saving Time: {save_xlam_duration:.2f} seconds")

    print(f"== XLAM Total Process: {time.time() - overall_xlam_start:.2f} seconds ==")
    traing_log.append(f"== XLAM Total Process: {time.time() - overall_xlam_start:.2f} seconds ==")

except Exception as e:
    print(f"An error occurred: {e}")
    print(f"Error detail: {str(e)}")
    traing_log.append(f"An error occurred: {e}")
    traing_log.append(f"Error detail: {str(e)}")
finally:
    # 세션 자동 종료
    print("Process finished. Unassigning runtime...")
    traing_log.append("Process finished. Unassigning runtime...")
    mnt_path = "/content/drive/MyDrive"
    folder_path = os.path.join(mnt_path, "TANGO2 - SDF", "코드", "20260409_xLAM2-8b-fc-r_26213-row_20percent-action_min", "02_모델_학습")
    with open(os.path.join(folder_path, "training_log.txt"), "w") as f:
        for line in traing_log:
            f.write(line + "\n")
    runtime.unassign()

Starting fine-tuning for XLAM...


Step,Training Loss,Validation Loss
150,0.306396,0.199274
300,0.178098,0.170369
450,0.157492,0.156210
600,0.147484,0.147589
750,0.140046,0.138991
900,0.133242,0.136145
1050,0.129852,0.132104
1200,0.127441,0.129123
1350,0.122546,0.128557
1500,0.121448,0.126500


>>> XLAM Training Time: 44932.59 seconds
Saving XLAM model to Drive...
>>> XLAM Saving Time: 0.42 seconds
== XLAM Total Process: 44933.01 seconds ==
Process finished. Unassigning runtime...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/TANGO2 - SDF/코드/20260409_xLAM2-8b-fc-r_26213-row_20percent-action_min/02_모델_학습/training_log.txt'